# Introduction to Sciline

<h4><i>Data processing workflow management tool.</i></h4>

<h3><a href="https://scipp.github.io/sciline/">scipp.github.io/sciline/</a></h3>

<br>

<div class="alert alert-warning">

**It is worth taking your time going through this notebook, do not try to rush it.**

</div>

Sciline is an open-source library developed by ESS for managing and visualizing data processing workflows (sometimes called "pipelines").

It defines workflows as directed acyclic graphs (DAGs) where the nodes are inputs, intermediate results, or final results, and edges are dependencies between the nodes.

This has some benefits:

- Any (named) intermediate result in the pipeline can be computed.
- The dependencies between intermediate results can be visualized.
- Implementations of intermediate result can be replaced.
- Results that are expensive to compute can be cached.
- Certainty that the computed result has not been corrupted by running jupyter cells out of order.

<br><br>

## Terminology

A Sciline **workflow** (or "pipeline") is defined by a set of transformations or **providers** that specify what inputs are needed to compute one specific output quantity.

The input and output quantities of the providers are called **domain types**.

A provider is a python function with [type annotations](../1-python/python_basics/intermediate_topics.ipynb#type-hints):

```python
# Example provider
def load_run(
    run_number: RunNumber,
    proposal_number: ProposalNumber,
    data_dir_path: DataPath,
) -> LoadedNexusData:

    filename = f'{proposal_number}_{run_number:06d}.hdf'
    path = os.path.join(data_dir_path, filename)

    data = scippnexus.load(path)
        
    return data
```

In the above example, to compute the `LoadedNexusData` quantity the `RunNumber`, `ProposalNumber` and the `DataPath` quantities are needed.

`LoadedNexusData`, `RunNumber`, `ProposalNumber` and `DataPath` are "domain types".

The domain types are the nodes in the workflow graph, and they represent the inputs, intermediate results, or final results of the workflow.

<br>

### Inputs, intermediate results, and final results

A typical data reduction workflow has some "**Inputs**", some "**Intermediate results**" and some "**Final results**".

Sciline does not distinguish between those, but it is useful to make a loose distinction:

**Inputs** are typically:

- The name of one or more NeXus files.
- Parameters defining a region of interest (ROI),
  - for example the wavelength range.
- The number of histogram bins.

**Intermediate results** are typically:

- List of events with associated coordinates, masks, and weights.
  - "Coordinates" such as wavelength, scattering angle, etc.
- Monitor wavelength histogram.
- Various calibration factors that are computed or loaded from file.

**Final results** are typically:

- Curve describing the scattering cross section $S(Q)$ as a function of momentum transfer (SANS).
- List of peaks and associated intensities (diffraction).
- Etc.

![Sciline graph example](images/sciline-graph-example.svg "Illustration of a Sciline workflow graph")

### How does Sciline know how to build the graph?

Each domain type is **unique**, and given a list of all functions to be used in the workflow,
Sciline can figure out how to connect the nodes in the graph.

Think of it like puzzle pieces that fit into each other:

<img src="images/sciline-puzzle-1.svg" width="600">

<br><br><br>

<img src="images/sciline-puzzle-2.svg" width="600">

## Example

In [ ]:
import os
from typing import NewType

import sciline as sl
import scipp as sc

from scippneutron.conversion.graph.beamline import beamline
from scippneutron.conversion.graph.tof import elastic

import sans_utils as utils

graph = {**beamline(scatter=True), **elastic("tof")}

### Creating a pipeline

We start by by defining the **domain types**:
the quantities representing input parameters, intermediate results and the final results of the pipeline.

In [ ]:
Foldername = NewType("Foldername", str)
"""Folder name for measurements."""

RawData = NewType("RawData", sc.DataArray)
"""Raw loaded data."""

CoordTransformGraph = NewType("CoordTransformGraph", dict)
"""Graph describing coordinate transformations."""

WavelengthData = NewType("WavelengthData", sc.DataArray)
"""Data with wavelength coordinate."""

QData = NewType("Qdata", sc.DataArray)
"""Data with Q coordinate."""

QBins = NewType("QBins", sc.Variable)
"""Bin edges in the Q dimension."""

QHistogram = NewType("QHistogram", sc.DataArray)
"""Data histogrammed in Q bins."""

Next, we create the 'provider' functions: the operations that determine how do we get from one domain type to the next.

In [ ]:
def load(folder: Foldername) -> RawData:
    """Load raw data from file"""
    return utils.load_sans(folder)


def to_wavelength(data: RawData, graph: CoordTransformGraph) -> WavelengthData:
    """Compute wavelength for events"""
    return data.transform_coords("wavelength", graph=graph)


def to_Q(data: WavelengthData, graph: CoordTransformGraph) -> QData:
    """Compute Q for events"""
    return data.transform_coords("Q", graph=graph)


def to_histogram(events: QData, qbins: QBins) -> QHistogram:
    """Histogram data in Q bins"""
    return events.hist(Q=qbins)

Finally, we **build the workflow** (also known as a 'pipeline') by supplying the list of our functions defined above:

In [ ]:
workflow = sl.Pipeline(
    # List the providers that make up the workflow.
    (load, to_wavelength, to_Q, to_histogram),
)

workflow.visualize(graph_attr={"rankdir": "LR"})

Some domain types are visualized in red color and with dashed border, those are the domain types that **lack a definition**.

### Setting parameters

To set a parameter on the workflow, we use the setitem operator `[]` (as one would do with a python dict):

In [ ]:
workflow[QBins] = sc.linspace("Q", 5.0e-3, 0.19, 201, unit="1/angstrom")
workflow[Foldername] = utils.fetch_data("3-mcstas/SANS_with_sample_many_neutrons")
workflow[CoordTransformGraph] = graph

We can now visualize the workflow again and see that the red boxes have now turned black;
all parameters required to compute the `QHistogram` have been set.

In [ ]:
workflow.visualize(graph_attr={"rankdir": "LR"})

### Computing quantities

Until now, not computations have taken place.
We have simply set-up the framework for the reduction in the shape of a task-graph, but nothing has been computed yet.

Building and manipulating the graph is very cheap, and only once we are satisfied with it should we then proceed with computing the final result.
This is done by calling the `compute` method:

In [ ]:
q_hist = workflow.compute(QHistogram)
q_hist

In [ ]:
q_hist.plot()

#### Intermediate results

One of the most powerful features about `sciline` is that once we have built a graph,
any step (node) in the graph can be inspected.

This is extremely useful for looking at intermediate results, while debugging a workflow for instance.
Computing intermediate results is also done with the `compute` method, but we simply pass as the argument the type that corresponds to the quantity we wish to compute.

In [ ]:
wavelength_data = workflow.compute(WavelengthData)
wavelength_data

#### Computing multiple results

It is also possible to request more than one result in one go.

This is useful because whenever we compute a quantity, `sciline` walks the entire graph all over again; nothing is cached.
This default behaviour ensures the correctness and reproducibility of results.

So when we first computed the `QHistogram` and then the `WavelengthData` above in 2 separate `compute` calls,
we actually loaded the file twice.

To avoid this, we can request both targets inside the call to compute, and the file will only be loaded once
(the scheduler will identify which parts of the graph are needed for which result and will not repeat branches used by both).

In [ ]:
two_results = workflow.compute((WavelengthData, QHistogram))
two_results

The results are stored in a dictionary where the keys are just the requested types:

In [ ]:
two_results[QHistogram]

## Common errors

In this section, we walk the reader through some of the most common errors that are encountered while working with `sciline`,
and try to bring extra explanations to the error messages, which are not always simple to understand.

### UnsatisfiedRequirement

This is probably the most common error you will encounter; and it typically means you have forgotten to set a parameter on the workflow.

If we go back to our original workflow where we still had the `Foldername` and `QBins` red boxes

In [ ]:
workflow = sl.Pipeline(
    # List the providers that make up the workflow.
    (load, to_wavelength, to_Q, to_histogram),
)

and we immediately try to compute the final histogram

In [ ]:
workflow.compute(QHistogram)

we are told that a requirement `Foldername` is missing to be able to compute the result.

Sciline stops at the first missing requirement.

### A missing provider

We build the same workflow, be here we forget to add one of the provider functions at build time:

In [ ]:
workflow = sl.Pipeline(
    # Missing load provider!
    (to_wavelength, to_Q, to_histogram),
)

# Set the parameters as before
workflow[QBins] = sc.linspace("Q", 5.0e-3, 0.19, 201, unit="1/angstrom")
workflow[Foldername] = utils.fetch_data("3-mcstas/SANS_with_sample_many_neutrons")
workflow[CoordTransformGraph] = graph

# The graph looks different
workflow.visualize(graph_attr={"rankdir": "LR"})

In [ ]:
workflow.compute(QHistogram)

We are now in a split state where the error message is telling it is missing the `RawData`.

The workflow is supposed to get the `RawData` from loading the data in the folder, but because provider function linking `Foldername` to `RawData` is missing,
the chain is broken and the `Foldername` is just an isolated box, not connected to any other parts of the graph.

### Long error messages

Sometimes, the error messages produced can appear dauntingly long.

A common example is when something goes wrong inside one of the provider functions.

The order in which the functions in the graph are called is managed/orchestrated by an external library called [Dask](https://www.dask.org/),
the calls happen deeply embedded in some scheduling mechanism that tries to optimize thread and memory use.

This means that when something fails inside one of the functions, the traceback will also contain a lot of Dask-specific information which users don't actually need read or care about.
Instead, one should scroll all the way to the bottom to read `CoordError: Coordinate 'q' not found.`

In [ ]:
def bad_to_histogram(events: QData, qbins: QBins) -> QHistogram:
    """Histogram data in Q bins"""
    # Wrong coordinate name: it should be uppercase Q!
    return events.hist(q=qbins)


workflow = sl.Pipeline(
    (
        load,
        to_wavelength,
        to_Q,
        bad_to_histogram,
    ),
)

workflow[Foldername] = utils.fetch_data("3-mcstas/SANS_with_sample_many_neutrons")
workflow[CoordTransformGraph] = graph
workflow[QBins] = 200

# Uncomment the next line to see the exception
workflow.compute(QHistogram)

## Generic domain types

Sometimes we want to replicate parts of a workflow and apply it to a different input.

A typical case is when we have a sample measurement and want to correct it by a background measurement.

In that case many of the processing steps are identical, but ultimately we want to subtract the background measurement from the sample measurement.

Generic domain types lets us define domain types that represent "Y of the X" such as:
- `Filename[Background]`: Filename of the background run.
- `QHistogram[Sample]`: The Q-histogram of the sample run.
- `QHistogram[Background]`: The Q-histogram of the background run.
- etc


In [ ]:
from typing import TypeVar


# Define concrete RunType values we will use.
Sample = NewType("Sample", int)
Background = NewType("Background", int)

# Define generic domain types
RunType = TypeVar("RunType", Sample, Background)

# Domain types

# We use sciline.Scope to make Filename a "generic" domain type that depends on RunType.
class Foldername(sl.Scope[RunType, str], str): ...

class RawData(sl.Scope[RunType, sc.DataArray], sc.DataArray): ...

class WavelengthData(sl.Scope[RunType, sc.DataArray], sc.DataArray): ...

class QData(sl.Scope[RunType, sc.DataArray], sc.DataArray): ...

class QHistogram(sl.Scope[RunType, sc.DataArray], sc.DataArray): ...

FinalHistogram = NewType("FinalHistogram", sc.DataArray)
"""Data histogrammed in Q bins where background has been subtracted"""


# Note that the QBins and coordinate transform graph are the same for
# Sample and Background runs, so there is no need to make them into
# generic types.

CoordTransformGraph = NewType("CoordTransformGraph", dict)

QBins = NewType("QBins", sc.Variable)



def load(folder: Foldername[RunType]) -> RawData[RunType]:
    """Load raw data from file"""
    return utils.load_sans(folder)


def to_wavelength(data: RawData[RunType], graph: CoordTransformGraph) -> WavelengthData[RunType]:
    """Compute wavelength for events"""
    return data.transform_coords("wavelength", graph=graph)


def to_Q(data: WavelengthData[RunType], graph: CoordTransformGraph) -> QData[RunType]:
    """Compute Q for events"""
    return data.transform_coords("Q", graph=graph)


def to_histogram(events: QData[RunType], qbins: QBins) -> QHistogram[RunType]:
    """Histogram data in Q bins"""
    return events.hist(Q=qbins)

# A new function that will subtract background from sample run.
# Note here that instead of [RunType], we have [Sample] and [Background]
def subtract_background(
    sample_data: QHistogram[Sample],
    background_data:QHistogram[Background],
) -> FinalHistogram:
    """Subtract background signal"""
    return sample_data - background_data


workflow = sl.Pipeline(
    # List the providers that make up the workflow.
    (load, to_wavelength, to_Q, to_histogram, subtract_background),
)

workflow.visualize(graph_attr={"rankdir": "LR"})

To set the parameter, we **must not forget** to also use the `[Sample]` and `[Background]` specifiers inside the square brackets:

In [ ]:
workflow[Foldername[Sample]] = utils.fetch_data("3-mcstas/SANS_with_sample_many_neutrons")
workflow[Foldername[Background]] = utils.fetch_data("3-mcstas/SANS_without_sample_many_neutrons")
workflow[QBins] = sc.linspace("Q", 5.0e-3, 0.19, 201, unit="1/angstrom")
workflow[CoordTransformGraph] = graph

And we can finally compute the final result:

In [ ]:
workflow.compute(FinalHistogram)

To compute an intermediate result, we also need to use the `[Sample]` and `[Background]` specifiers:

In [ ]:
workflow.compute(QData[Background])

## How will Sciline be used at ESS?

Most instruments will have **one or more** associated Sciline workflows.

The workflows will be the basic interface to the data reduction software.

On top of that interface we can build simpler but less flexible interfaces.

- But that will take time.
- In the early days after HC the interface to the data reduction will be mainly in the form of Sciline workflows.

### What do I need to know?

1. How to figure out **what quantity to compute** with the workflow.
   - Look at the workflow graph and read on the technique package documentation page.
3. How to figure out **what parameters are needed** to compute the target quantity.
   - Error messages tell you what is missing, or you can look at the workflow graph.
5. **How to set parameters** on the workflow.
7. **How to compute** the desired quantity.
8. How to read and **understand common error messages**.


### Now you are ready to proceed to the "Sciline workflow" section in the exercises!